# Alternative hybrid methods for scalar diffusion

Compare Robin MH, three-field MH²M, MsHHO and residual Petrov–Galerkin MHM on
small affine analytical patches. These methods retain different interface
unknowns. Robin multipliers, conormal variables, face pressure moments and
physical normal Darcy flux are identified explicitly in each record.

Local/face compatibility and the method-specific degree conditions remain part
of each method's contract. Passing an affine patch does not certify arbitrary
combinations of local and trace spaces.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/06ac6856e58f3d6386e89c4e8f822499dcf91dda16fd424b0f848921d86fccd9/hybrid_methods-companion.zip"
COMPANION_SHA256 = "06ac6856e58f3d6386e89c4e8f822499dcf91dda16fd424b0f848921d86fccd9"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/hybrid_methods.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


## Declare the local and global equations

The three-field formulation keeps pressure and outward conormal coordinates locally. Its global variable is the continuous scalar trace. The conormal is minus the physical Darcy flux.

$$
\begin{aligned}
A_K p_K-B_K\eta_K &= f_K,\\
-B_K^T p_K+P_K\rho &= 0,\\
\sum_K P_K^T\eta_K &= 0.
\end{aligned}
$$

The example supplies these blocks to `LocalEquations`; its source and boundary values are application data. Inspect the actual local definition below.


In [ ]:
from inspect import getsource
from examples.hybrid_formulation import build_problem, three_field_cell, recover_pressure, pressure
from pymhm import assemble

print(getsource(three_field_cell))
three_field = assemble(build_problem("three-field"))
three_field_solution = three_field.solve()
three_field_pressure = recover_pressure(three_field, three_field_solution)
three_field_error = max(
    np.max(abs(values - pressure(data["points"])))
    for values, data in zip(three_field_pressure, three_field.local_metadata, strict=True)
)
print({"pressure_error": float(three_field_error), "residual": three_field_solution.raw_residual})
assert three_field_error < 1e-10


## Define a moment reconstruction

For MsHHO, the user declares physical cell and face moments, and the package computes the constrained energy reconstruction. Face variables are pressure integrals. The example then declares the resulting local cell/face energy blocks and lets the same generic assembler eliminate cell moments.

$$
\begin{aligned}
A_K R_K+C_K\mu_K &= 0,\\
C_K^T R_K &= I,\\
E_K &= R_K^T A_K R_K.
\end{aligned}
$$

This is an affine analytical patch with local P2 pressure, P0 face moments and one cell integral. It is not a literature convergence study.


In [ ]:
from examples.hybrid_formulation import moment_cell

print(getsource(moment_cell))
moment_system = assemble(build_problem("moments"))
moment_solution = moment_system.solve()
moment_pressure = recover_pressure(moment_system, moment_solution)
moment_error = max(
    np.max(abs(values - pressure(data["points"])))
    for values, data in zip(moment_pressure, moment_system.local_metadata, strict=True)
)
print({"pressure_error": float(moment_error), "residual": moment_solution.raw_residual})
assert moment_error < 1e-10


## Compare the preserved method implementations

These additional checks use the compatibility implementations with their original spaces and conventions. They preserve the wider numerical method inventory; the primary examples above define their own blocks through the variational API.


In [ ]:
from examples.tutorial_scalar_variants import (
    VARIANTS, affine_pressure, affine_flux, run_variant,
)
selected_methods = ('mh-robin', 'mh-robin3d', 'mh2m', 'mh2m3d', 'mshho', 'mshho3d', 'mshho-face', 'pgmhm')
records = [run_variant(name) for name in selected_methods]
for record in records:
    print(record["variant"], record["spaces_and_conventions"], record["metrics"])


## Inspect one method before changing its spaces

Use `VARIANTS` to inspect the approximation and boundary convention. The
reconstruction, conservation diagnostics and algebraic residual remain separate.
The detailed studies for these methods are the numbered notebooks in this folder.


In [ ]:
method = "mh2m"
print(VARIANTS[method])
print(next(record for record in records if record["variant"] == method))
